## EAR (Eye Aspect Ratio) image processing function... 

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import os
import csv

mp_face = mp.solutions.face_mesh
face_mesh = mp_face.FaceMesh(static_image_mode=True, max_num_faces=1)

# MediaPipe indices for eyes (approximate, can be refined)
LEFT_EYE = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [263, 387, 385, 362, 380, 373]

def euclidean(p1, p2):
    return np.linalg.norm(np.array(p1) - np.array(p2))

def compute_ear(landmarks, eye_indices):
    A = euclidean(landmarks[eye_indices[1]], landmarks[eye_indices[5]])
    B = euclidean(landmarks[eye_indices[2]], landmarks[eye_indices[4]])
    C = euclidean(landmarks[eye_indices[0]], landmarks[eye_indices[3]])
    return (A + B) / (2.0 * C) if C != 0 else 0

def process_image(image_path):
    image = cv2.imread(image_path)
    if image is None:
        return None  # skip broken images

    h, w, _ = image.shape
    results = face_mesh.process(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))

    if results.multi_face_landmarks:
        lm = results.multi_face_landmarks[0].landmark
        coords = [(int(pt.x * w), int(pt.y * h)) for pt in lm]

        left_ear = compute_ear(coords, LEFT_EYE)
        right_ear = compute_ear(coords, RIGHT_EYE)
        return (left_ear + right_ear) / 2.0
    return None